# Thí nghiệm tiền xử lý (A–F) bằng GPU Colab

Dùng cho cả bộ video ổ gà và bộ ảnh 3 lớp: chọn bằng biến `SUITE` ở bước 2.

Notebook chạy cùng code, cùng seed và cùng tham số train như trên máy; chỉ khác là train bằng GPU.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`, sau đó `Runtime → Run all`.

Model và kết quả được ghi thẳng vào Google Drive (`MyDrive/Digital_Image_Processing/outputs/`).
Nếu Colab bị ngắt kết nối, chỉ cần `Run all` lại: thí nghiệm đã xong được giữ nguyên,
thí nghiệm đang dở chạy tiếp từ `last.pt`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Kết nối Google Drive

Đổi `DRIVE_DIR` nếu bạn để `colab_package.zip` ở thư mục khác.

`SUITE` phải khớp với lệnh đóng gói `python -m src.utils.package_for_colab --suite ...`:
- Video ổ gà: `configs/video_preprocessing_experiments.yaml`
- Ảnh 3 lớp (pothole, crack, manhole): `configs/preprocessing_experiments.yaml`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Digital_Image_Processing'
ZIP_PATH = f'{DRIVE_DIR}/colab_package.zip'
PROJECT = '/content/project'
SUITE = 'configs/video_preprocessing_experiments.yaml'  # 3 lớp: 'configs/preprocessing_experiments.yaml'
import os
assert os.path.isfile(ZIP_PATH), f'Không thấy {ZIP_PATH}. Hãy tải colab_package.zip lên thư mục đó.'

## 3. Giải nén code và dữ liệu

In [ ]:
import os
if not os.path.isdir(f'{PROJECT}/src'):
    !mkdir -p {PROJECT}
    !unzip -q "{ZIP_PATH}" -d {PROJECT}
%cd {PROJECT}
!ls

## 4. Cài thư viện (cùng phiên bản Ultralytics với máy)

In [ ]:
!pip install -q ultralytics==8.4.135
import ultralytics
print('ultralytics', ultralytics.__version__)

## 5. Cấu hình cho Colab

- Sửa đường dẫn trong `dataset.yaml` cho đúng thư mục trên Colab.
- Tạo bản cấu hình chạy GPU (`device: 0`). Mọi tham số train khác giữ nguyên, nên 6 thí nghiệm vẫn so sánh công bằng.
- Nối thư mục model của bộ thí nghiệm và `results` sang Google Drive để không mất kết quả khi bị ngắt.

In [ ]:
import shutil, yaml
from pathlib import Path
from src.data.dataset_utils import dataset_yaml_text

project = Path(PROJECT)
suite = yaml.safe_load((project / SUITE).read_text(encoding='utf-8'))
dataset = (project / suite['dataset']['source_yaml']).parent
assert (dataset / 'dataset.yaml').is_file(), f'Gói zip không có dữ liệu {dataset}: kiểm tra lại --suite khi đóng gói'
(dataset / 'dataset.yaml').write_text(dataset_yaml_text(dataset), encoding='utf-8')

suite['training']['device'] = 0
suite['training']['workers'] = 2
COLAB_SUITE = SUITE.replace('.yaml', '_colab.yaml')
(project / COLAB_SUITE).write_text(yaml.safe_dump(suite, sort_keys=False), encoding='utf-8')
RESULTS_DIR = suite['project'].get('results', 'results/experiments')

outputs = Path(DRIVE_DIR) / 'outputs'
model_dir = suite['project']['output']  # experiments/yolo_video hoặc experiments/yolo
for local, remote in ((model_dir, outputs / Path(model_dir).name), ('results', outputs / 'results')):
    remote.mkdir(parents=True, exist_ok=True)
    link = project / local
    if not link.is_symlink():
        if link.exists():
            shutil.rmtree(link)
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(remote, target_is_directory=True)
    print(link, '->', remote)
print('Bộ thí nghiệm:', SUITE, '| dữ liệu:', dataset.relative_to(project))

## 6. Chạy test (khoảng 30 giây)

In [ ]:
!python -m pytest -q

## 7. Chạy 6 thí nghiệm A_raw → F_combined

Trên T4 mất khoảng 30–60 phút. Màn hình hiện tiến độ từng epoch.

In [ ]:
!python -m src.detection.run_preprocessing_experiments --suite {COLAB_SUITE}

## 8. Xem kết quả

In [ ]:
import pandas as pd
table = pd.read_csv(f'{RESULTS_DIR}/experiment_results.csv')
display(table[['experiment', 'preprocessing', 'precision', 'recall', 'mAP50', 'mAP50_95',
               'pothole_mAP50', 'crack_mAP50', 'manhole_mAP50', 'preprocessing_ms', 'inference_ms', 'total_ms']])
print(open(f'{RESULTS_DIR}/experiment_analysis.md', encoding='utf-8').read())

## 9. Nén kết quả để tải về máy

Tạo `colab_results.zip` trong `MyDrive/Digital_Image_Processing/`. Tải file đó về và giải nén vào
thư mục dự án `D:\Digital_Image_Processing` (xem hướng dẫn trong README).

Lưu ý: thời gian xử lý (`*_ms`) đo trên GPU Colab, nhanh hơn trên máy của bạn.

In [ ]:
archive = shutil.make_archive(f'{DRIVE_DIR}/colab_results', 'zip', root_dir=outputs)
print('Đã lưu:', archive)